# 📘 智能体架构 16：细胞自动机 / 基于网格的系统

欢迎探索一种根本不同的智能体架构：**细胞自动机**和**基于网格的智能体系统**。该模式受自然复杂系统和康威生命游戏等概念的启发。它将范式从少数复杂、集中的智能体转变为在网格上运行的大量简单、去中心化的智能体。

在这个模型中，环境本身成为智能体。网格中的每个单元格都是一个迷你智能体，具有自己的状态和一组简单的规则，用于根据其直接邻居更新该状态。没有中央控制器或复杂的寻路算法。相反，智能的、全局行为**涌现**自这些简单局部规则的重复、同步应用。系统成为一个"计算织物"，通过类似波的信息传播来解决问题。

为了在详细、复杂的实现中演示这一点，我们将构建一个**仓库物流模拟器**。我们的目标是通过将物品从货架移动到包装站来完成订单。我们将不使用单一的"机器人"智能体来解决这个复杂的空间推理任务，而是通过编程网格单元本身来集体计算最佳路径。

### 定义
**基于网格的智能体系统**是一种架构，其中大量简单智能体（或"单元格"）排列在空间网格中。每个智能体都有一个状态，并基于一组仅考虑其直接邻居状态的规则同步更新该状态。复杂的、高级的模式和解决问题的能力从这些局部交互中涌现。

### 高层工作流程

1.  **网格初始化：** 创建单元格智能体网格，每个单元格初始化一个类型（例如，障碍物、空）和一个状态（例如，一个值）。
2.  **设置边界条件：** 为一个或多个单元格赋予特殊状态以开始计算（例如，"目标"单元格的值设置为 0）。
3.  **同步滴答：** 系统向前"滴答"。在每个滴答中，每个单元格同时根据其邻居的当前状态计算其下一个状态。
4.  **涌现：** 随着系统滴答，信息像波一样在网格上传播。这可以创建梯度、路径和其他复杂结构。
5.  **状态稳定：** 系统运行直到网格状态稳定（不再发生更改），表明计算完成。
6.  **读取：** 然后直接从网格的最终状态中读取问题的解决方案（例如，通过沿着计算的梯度）。

### 适用场景 / 应用
*   **空间推理与物流：** 动态环境中的最佳寻路（如我们的仓库示例）。
*   **复杂系统模拟：** 模拟具有涌现行为的现象，如森林火灾、疾病传播或城市增长。
*   **并行计算：** 某些算法可以映射到细胞自动机模型，以便在高度并行的硬件（如 GPU）上执行。

### 优缺点
*   **优点：**
    *   **高度并行性：** 逻辑本质上是并行的，使其在适当的硬件上非常快。
    *   **适应性：** 系统可以通过简单地重新传播其波来动态响应环境中的变化（例如，新障碍物）。
    *   **涌现复杂性：** 可以用令人惊讶的简单规则解决非常复杂的问题。
*   **缺点：**
    *   **设计复杂性：** 设计局部规则以产生所需的全球行为可能具有挑战性和不直观。
    *   **不太可解释：** 很难询问单个单元格"为什么"它具有某个状态；推理分布在整个系统中。

## 阶段 0：基础与环境设置

我们需要 `numpy` 进行高效的网格操作，`rich` 进行高质量的终端可视化。

In [ ]:
# !pip install -q -U langchain-anthropic rich python-dotenv numpy

In [ ]:
import os
import numpy as np
import time
from typing import List, Dict, Any, Optional, Tuple
from dotenv import load_dotenv
from IPython.display import clear_output

# LangChain for optional final summary
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

# For pretty printing and visualization
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

# --- API Key and Tracing Setup ---
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Cellular Automata"

required_vars = ["ANTHROPIC_API_KEY", "LANGCHAIN_API_KEY"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

## 阶段 1：构建细胞自动机环境

这是最关键的阶段。我们将为模拟定义两个核心类：
1.  `CellAgent`：表示网格中的单个单元格。它包含其类型、其状态（寻路值）和用于更新该状态的局部规则。
2.  `WarehouseGrid`：整个系统的容器。它将管理 `CellAgent` 的二维数组，编排同步 `tick` 更新，并处理可视化。

In [3]:
console = Console()

class CellAgent:
    """A single agent in our grid. Its only job is to update its value based on neighbors."""
    def __init__(self, cell_type: str, item: Optional[str] = None):
        self.type = cell_type # 'EMPTY', 'OBSTACLE', 'SHELF', 'PACKING_STATION'
        self.item = item
        self.pathfinding_value = float('inf')

    def update_value(self, neighbors: List['CellAgent']):
        """The core local rule: my new value is 1 + the minimum value of my non-obstacle neighbors."""
        if self.type == 'OBSTACLE':
            return float('inf')
        
        min_neighbor_value = float('inf')
        for neighbor in neighbors:
            if neighbor.pathfinding_value < min_neighbor_value:
                min_neighbor_value = neighbor.pathfinding_value
        
        # The +1 represents the cost of moving from a neighbor to this cell
        return min(self.pathfinding_value, min_neighbor_value + 1)

class WarehouseGrid:
    """Manages the entire grid of CellAgents and the simulation loop."""
    def __init__(self, layout: List[str]):
        self.height = len(layout)
        self.width = len(layout[0])
        self.grid = self._create_grid_from_layout(layout)
        self.item_locations = self._get_item_locations()

    def _create_grid_from_layout(self, layout):
        grid = np.empty((self.height, self.width), dtype=object)
        for r, row_str in enumerate(layout):
            for c, char in enumerate(row_str):
                if char == ' ':
                    grid[r, c] = CellAgent('EMPTY')
                elif char == '#':
                    grid[r, c] = CellAgent('OBSTACLE')
                elif char == 'P':
                    grid[r, c] = CellAgent('PACKING_STATION')
                else: # It's an item
                    grid[r, c] = CellAgent('SHELF', item=char)
        return grid

    def _get_item_locations(self) -> Dict[str, Tuple[int, int]]:
        locations = {}
        for r in range(self.height):
            for c in range(self.width):
                if self.grid[r, c].type == 'SHELF':
                    locations[self.grid[r, c].item] = (r, c)
                if self.grid[r, c].type == 'PACKING_STATION':
                    locations['P'] = (r, c)
        return locations

    def get_neighbors(self, r: int, c: int) -> List[CellAgent]:
        neighbors = []
        for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]: # N, S, E, W
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.height and 0 <= nc < self.width:
                neighbors.append(self.grid[nr, nc])
        return neighbors

    def tick(self) -> bool:
        """Performs one synchronous update of all cells. Returns True if any value changed."""
        changed = False
        # First, calculate all new values based on the current state
        new_values = np.empty((self.height, self.width))
        for r in range(self.height):
            for c in range(self.width):
                neighbors = self.get_neighbors(r, c)
                new_values[r, c] = self.grid[r, c].update_value(neighbors)
        
        # Then, apply all the new values
        for r in range(self.height):
            for c in range(self.width):
                if self.grid[r, c].pathfinding_value != new_values[r, c]:
                    self.grid[r, c].pathfinding_value = new_values[r, c]
                    changed = True
        return changed

    def visualize(self, show_values: bool = False, title: str = "Warehouse Grid"):
        """Displays the grid state using Rich."""
        table = Table(title=title, show_header=False, show_edge=True, padding=0)
        for _ in range(self.width):
            table.add_column(justify="center")
        
        for r in range(self.height):
            row_renderables = []
            for c in range(self.width):
                cell = self.grid[r, c]
                val = cell.pathfinding_value
                display_char = ''
                if cell.type == 'EMPTY': display_char = '[grey70]·[/grey70]'
                elif cell.type == 'OBSTACLE': display_char = '[red]█[/red]'
                elif cell.type == 'PACKING_STATION': display_char = '[bold green]P[/bold green]'
                elif cell.type == 'SHELF': display_char = f'[bold blue]{cell.item}[/bold blue]'

                if show_values and val != float('inf'):
                    # Color code the path values
                    color = int(255 - (val * 5) % 255)
                    row_renderables.append(f"[rgb({color},{color},{color}) on rgb(30,30,60)]{int(val):^3}[/]")
                else:
                    row_renderables.append(f" {display_char} ")
            table.add_row(*row_renderables)
        console.print(table)

print("Cellular Automata environment defined successfully.")

Cellular Automata environment defined successfully.


## 阶段 2：实现涌现行为

网格本身只是一个框架。我们需要实现使用细胞自动机解决问题的高级逻辑。这涉及两个关键的涌现行为：

1.  **路径波传播：** 一个函数，设置目标并让网格 `tick`，直到在整个仓库地板上形成完整的寻路梯度。
2.  **梯度下降遍历：** 一个函数，模拟"搬运工"智能体从物品货架开始，简单地沿着最陡峭下降路径（最低的 `pathfinding_value`）直到到达目标。

In [4]:
def propagate_path_wave(grid: WarehouseGrid, target_pos: Tuple[int, int], visualize_steps: bool = False):
    """Resets and then runs the simulation until the pathfinding values stabilize."""
    # Reset all pathfinding values
    for r in range(grid.height):
        for c in range(grid.width):
            grid.grid[r, c].pathfinding_value = float('inf')
            
    # Set the target's value to 0 to start the wave
    grid.grid[target_pos].pathfinding_value = 0
    
    tick_count = 0
    while True:
        tick_count += 1
        if visualize_steps:
            clear_output(wait=True)
            grid.visualize(show_values=True, title=f"Path Wave Propagation (Tick #{tick_count})")
            time.sleep(0.1)
        
        changed = grid.tick()
        if not changed:
            break
    if visualize_steps:
        clear_output(wait=True)
        grid.visualize(show_values=True, title=f"Path Wave Propagation (Stabilized at Tick #{tick_count})")

def trace_and_move_item(grid: WarehouseGrid, start_pos: Tuple[int, int]) -> List[Tuple[int, int]]:
    """Follows the gradient from the start position back to the target (value 0)."""
    path = [start_pos]
    r, c = start_pos
    
    while grid.grid[r, c].pathfinding_value > 0:
        neighbors = grid.get_neighbors(r, c)
        best_neighbor_pos = None
        min_val = grid.grid[r, c].pathfinding_value
        
        # Find the neighbor with the lowest pathfinding value
        for neighbor_cell in neighbors:
            # Find the position of the neighbor cell
            pos_list = np.where(grid.grid == neighbor_cell)
            if len(pos_list[0]) > 0:
                nr, nc = pos_list[0][0], pos_list[1][0]
                if neighbor_cell.pathfinding_value < min_val:
                    min_val = neighbor_cell.pathfinding_value
                    best_neighbor_pos = (nr, nc)
        
        if best_neighbor_pos:
            path.append(best_neighbor_pos)
            r, c = best_neighbor_pos
        else:
            console.print("[red]Error: Path tracing got stuck. No downhill neighbor found.[/red]")
            break
            
    return path

print("Emergent behavior functions defined successfully.")

Emergent behavior functions defined successfully.


## 阶段 3：完整编排工作流程

现在我们将创建模拟整个订单履行流程的顶级函数。这将演示如何组合涌现行为来解决多步骤问题。

In [ ]:
def fulfill_order(layout: List[str], order: List[str], visualize_waves: bool = False):
    """The main orchestration function."""
    grid = WarehouseGrid(layout)
    console.print("--- Initial Warehouse State ---")
    grid.visualize()
    
    packing_station_pos = grid.item_locations['P']
    
    for i, item_id in enumerate(order):
        panel_title = f"[bold]Step {i+1}: Fulfill Item '{item_id}'[/bold]"
        log_messages = []
        
        item_pos = grid.item_locations.get(item_id)
        if not item_pos:
            console.print(Panel(f"[red]Error: Item '{item_id}' not found in warehouse.[/red]", title=panel_title))
            continue
            
        # 1. Compute the path wave from the packing station
        log_messages.append("🌊 Computing path wave from Packing Station...")
        propagate_path_wave(grid, packing_station_pos, visualize_steps=visualize_waves)
        
        # 2. Trace the path for the current item
        log_messages.append(f"🚚 Found path for item {item_id}. Moving along gradient...")
        path = trace_and_move_item(grid, item_pos)
        path_str = ' -> '.join(map(str, path))
        log_messages.append(f"Path: {path_str}")

        # 3. Update the grid state (item is now at packing station)
        grid.grid[item_pos].type = 'EMPTY'
        grid.grid[item_pos].item = None
        log_messages.append(f"✅ Item '{item_id}' has been moved to the packing station.")
        console.print(Panel('\n'.join(log_messages), title=panel_title, border_style="blue"))
        
    console.print(Panel(f"The system successfully fulfilled the order for items {order} by emergently computing paths through local cell interactions.", title="[bold green]🎉 Order Fulfillment Complete![/bold green]", border_style="green"))
    return grid

# --- Main Execution ---
warehouse_layout = [
    "#######",
    "# D   #",
    "# ### #",
    "#A#C# #",
    "# # #B#",
    "#  P  #",
    "#######",
]
order_to_fulfill = ['A', 'B']
final_grid = fulfill_order(warehouse_layout, order_to_fulfill, visualize_waves=True)

# --- Optional: LLM Interpretation ---
console.print("\n--- 🤖 LLM Interpretation of the Final State ---")
model = os.environ.get("MODEL_NAME", "claude-opus-4-5-20251101")
base_url = os.environ.get("BASE_URL")
llm = ChatAnthropic(model=model, base_url=base_url, temperature=0.2)
summary_prompt = ChatPromptTemplate.from_template("You are a logistics manager. Briefly summarize the outcome of the following order fulfillment report.\n\nOrder: {order}\nFinal Warehouse State: All items from the order have been moved to the packing station. Items A and B were retrieved. Original locations were {loc_A} and {loc_B}. The floor is now clear.")
summary_chain = summary_prompt | llm
final_summary = summary_chain.invoke({
    "order": order_to_fulfill, 
    "loc_A": WarehouseGrid(warehouse_layout).item_locations['A'],
    "loc_B": WarehouseGrid(warehouse_layout).item_locations['B']
}).content
console.print(Markdown(final_summary))

### 结果分析

这个详细的实现完美地展示了细胞自动机解决问题的独特性质：

1.  **没有中央规划者：** 我们从未使用过像 A* 这样的全局寻路算法。我们从未以自顶向下的方式计算路径。最佳路径是网格本身的*涌现属性*。

2.  **信息即波：** `propagate_path_wave` 函数是关键。可视化显示了距离包装站的距离如何逐滴答地在网格上传播，自然地绕过障碍物。这是"计算织物"在工作。网格本质上同时计算了*每个单独空方块*到包装站的最短路径。

3.  **简单智能体，复杂行为：** 移动物品的"搬运工"非常简单。它唯一的逻辑是"找到数字最低的邻居并移动到那里。"所有复杂的环境推理已经通过路径波编码到网格状态中。

4.  **适应性：** 如果我们要通过添加新障碍物来改变仓库布局，我们不需要重写复杂的寻路算法。我们将简单地重新运行波传播，路径值将自动正确地绕过新障碍物流动，展示了系统固有的适应性。

这是从传统智能体设计的根本转变。我们不是构建一个导航愚蠢环境的智能智能体，而是构建一个由许多愚蠢智能体组成的智能环境，这些智能体集体解决问题。

## 结论

在本笔记本中，我们构建了一个完全实现的**细胞自动机 / 基于网格的智能体系统**。我们超越了理论，实现了一个复杂空间推理问题（仓库物流）的实际解决方案。

我们已经亲眼看到复杂的、目标导向的行为如何从网格上迷你智能体之间简单、局部规则的同步执行中涌现。**波传播**和**梯度下降**的概念不是以自顶向下的方式明确编程的，而是细胞自动机演化的自然结果。

虽然该架构并不适合所有问题，但对于涉及动态环境中的空间推理、模拟和优化的任务，它异常强大。它鼓励我们将智能体系统不视为单个"机器人"，而是视为一个**可编程的计算环境**，可以配置为以大规模并行和自适应的方式解决问题。